# Working with Dates and Time in Pandas

Raw date/time columns almost always need to be broken apart before a model can use them — a model can't directly learn from `'2019-06-01'`, but it can learn from separate `year`, `month`, `day_of_week`, `is_weekend` columns pulled out of it. This notebook covers that, on two datasets: `orders.csv` (date only) and `messages.csv` (date + time).

## 1. Setup and Data

In [1]:
import numpy as np
import pandas as pd

In [2]:
date = pd.read_csv('orders.csv')
time = pd.read_csv('messages.csv')

FileNotFoundError: [Errno 2] No such file or directory: 'orders.csv'

In [ ]:
date.head()

In [ ]:
time.head()

`.info()` confirms the `date` columns are currently plain **strings** (`object` dtype) — before any `.dt` accessor works, they need converting to an actual datetime type first.

In [ ]:
date.info()

In [ ]:
time.info()

# Working with Dates

`pd.to_datetime()` parses the string column into pandas' `datetime64` dtype — confirmed below by `.info()` showing the changed dtype. Every `.dt.xxx` accessor used from here on only works on a column already converted this way.

In [ ]:
# Converting to datetime datatype
date['date'] = pd.to_datetime(date['date'])

In [ ]:
date.info()

#### 1. Extract year

`.dt.year` pulls the year out as a plain integer column.

In [ ]:
date['date_year'] = date['date'].dt.year

date.sample(5)

#### 2. Extract Month

`.dt.month` gives the month as a number (1–12); `.dt.month_name()` gives the readable name (`'June'`) — useful for plots and reports, `.dt.month` for anything doing numeric comparisons or sorting.

In [ ]:
date['date_month_no'] = date['date'].dt.month

date.head()

In [ ]:
date['date_month_name'] = date['date'].dt.month_name()

date.head()

#### Extract Days

In [ ]:
date['date_day'] = date['date'].dt.day

date.head()

`.dt.dayofweek` gives Monday=0 through Sunday=6 as a plain number — handy for a model, but not very readable on its own, which is why `.dt.day_name()` (below) is often extracted alongside it.

In [ ]:
# day of week
date['date_dow'] = date['date'].dt.dayofweek

date.head()


In [ ]:
# day of week - name

date['date_dow_name'] = date['date'].dt.day_name()

date.drop(columns=['product_id','city_id','orders']).head()

`np.where(condition, 1, 0)` is a compact way to build a binary flag column — here, whether the order fell on a weekend. This kind of derived flag is often more directly useful to a model than the raw day name.

In [ ]:
# is weekend?

date['date_is_weekend'] = np.where(date['date_dow_name'].isin(['Sunday', 'Saturday']), 1,0)

date.drop(columns=['product_id','city_id','orders']).head()

#### Extract week of the year

### ⚠️ Important: `.dt.week` Is Removed in Current Pandas

`.dt.week` (and `.dt.weekofyear`) were deprecated back in pandas 1.1 and **fully removed in pandas 2.0** — this line now raises `AttributeError: 'DatetimeProperties' object has no attribute 'week'`. The replacement is `.dt.isocalendar().week`, which returns the ISO week number (1–52/53).

In [ ]:
date['date_week'] = date['date'].dt.isocalendar().week

date.drop(columns=['product_id', 'city_id', 'orders']).head()


#### Extract Quarter

`.dt.quarter` gives which quarter of the year (1–4) the date falls in.

In [ ]:
date['quarter'] = date['date'].dt.quarter

date.drop(columns=['product_id','city_id','orders']).head()

#### Extract Semester

No built-in `.dt.semester` exists — but it's simple to derive from the quarter already extracted: quarters 1–2 make up the first half of the year (semester 1), quarters 3–4 the second (semester 2).

In [ ]:
date['semester'] = np.where(date['quarter'].isin([1,2]), 1, 2)

date.drop(columns=['product_id','city_id','orders']).head()

#### Extract Time elapsed between dates

`datetime.datetime.today()` gets the current date/time at the moment the cell runs — the reference point for every "time elapsed since" calculation below.

In [ ]:
import datetime

today = datetime.datetime.today()

today

In [ ]:
today - date['date']

Subtracting one datetime column from another produces a **`Timedelta`** — a duration, not a date. Printed directly it shows as `'X days HH:MM:SS'`, which is informative but not a plain number a model can use directly.

In [ ]:
(today - date['date']).dt.days

`.dt.days` extracts just the whole-number day count from that duration.

### A Caveat on "Months Passed"

Dividing by `np.timedelta64(1, 'M')` treats a month as a **fixed average length** (~30.44 days) — it isn't aware of calendar months actually varying between 28 and 31 days, so treat this as an approximation, not an exact calendar-month count.

In [ ]:
# Months passed

np.round((today -date['date']) / np.timedelta64(1, 'M'),0)

### Visualizing Orders Over Time

With `date_month_name` and `date_dow_name` already extracted, it's easy to check whether orders show any obvious weekly or monthly pattern — exactly the kind of signal these engineered columns are meant to expose to a model.

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

date.groupby('date_dow_name')['orders'].sum().reindex(
    ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
).plot(kind='bar', ax=axes[0])
axes[0].set_title('Total Orders by Day of Week')

date.groupby('date_month_name')['orders'].sum().plot(kind='bar', ax=axes[1], color='#DD8452')
axes[1].set_title('Total Orders by Month')

plt.tight_layout()
plt.show()


## 2. Working with Time — `messages.csv`

Same `pd.to_datetime()` conversion first, then time-specific fields (hour, minute, second) rather than date-specific ones.

In [ ]:
time.info()

In [ ]:
# Converting to datetime datatype
time['date'] = pd.to_datetime(time['date'])

In [ ]:
time.info()

In [ ]:
time['hour'] = time['date'].dt.hour
time['min'] = time['date'].dt.minute
time['sec'] = time['date'].dt.second

time.head()

`.dt.hour` / `.dt.minute` / `.dt.second` each pull out that one component as a plain integer — useful for, say, checking what time of day messages tend to be sent.

#### Extract Time part

In [ ]:
time['time'] = time['date'].dt.time

time.head()

`.dt.time` returns just the time-of-day portion (no date) as a Python `time` object — handy when the date part is irrelevant and only the clock time matters.

#### Time difference

In [ ]:
today - time['date']

In [ ]:
# in seconds

(today - time['date'])/np.timedelta64(1,'s')

Same duration, converted into different units by dividing by the matching `timedelta64` unit.

In [ ]:
# in minutes

(today - time['date'])/np.timedelta64(1,'m')

In [ ]:
# in hours

(today - time['date'])/np.timedelta64(1,'h')

## Summary

| Task | Method |
|---|---|
| Convert a string column to datetime | `pd.to_datetime(col)` |
| Extract year / month / day | `.dt.year`, `.dt.month`, `.dt.day` |
| Readable month/day names | `.dt.month_name()`, `.dt.day_name()` |
| Day of week as a number | `.dt.dayofweek` (Mon=0 ... Sun=6) |
| Week of year | `.dt.isocalendar().week` *(not the removed `.dt.week`)* |
| Quarter | `.dt.quarter` |
| Build a binary flag column | `np.where(condition, 1, 0)` |
| Duration between two dates | `date2 - date1` → `.dt.days`, or `/ np.timedelta64(1, unit)` for other units |
| Extract hour/minute/second | `.dt.hour`, `.dt.minute`, `.dt.second` |
| Just the time-of-day part | `.dt.time` |

### One thing to watch for
`np.timedelta64(1, 'M')` uses an *average* month length, not real calendar months — fine for rough features, not for anything needing exact calendar-month boundaries.